# Week 9 - Activity 1: Analyzing Sparse Autoencoder Features

In this activity, we'll explore the sparse autoencoder features from ["Towards Monosemanticity: Decomposing Language Models With Dictionary Learning"](https://transformer-circuits.pub/2023/monosemantic-features/). We'll:

1. Load pre-trained sparse autoencoder features
2. Analyze feature activations on different inputs
3. Visualize and interpret feature patterns
4. Find interesting or surprising features

This helps us understand how language models represent information internally.

In [1]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats

## 1. Load Pre-trained Features

We'll use the pre-trained sparse autoencoder features from the paper:

In [ ]:


# Load model
model_name = "gpt2"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Get model dimensions
hidden_size = model.config.hidden_size
n_features = hidden_size  # Use same number of features as hidden size

# Create random feature directions (this is a simplified approach)
feature_directions = np.random.randn(hidden_size, n_features)
feature_directions = feature_directions / np.linalg.norm(feature_directions, axis=0)

# Create feature metadata
feature_metadata = pd.DataFrame({
    'type': ['unknown'] * n_features,
    'description': ['No description available'] * n_features
})

print(f"Number of features: {len(feature_metadata)}")
print(f"Feature direction shape: {feature_directions.shape}")



## 2. Feature Activation Analysis

Let's analyze how different features activate on various inputs:

In [ ]:


def get_feature_activations(text: str, model, tokenizer, feature_directions) -> np.ndarray:
    """Get feature activations for a given input text."""
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    # Get activations from the last layer
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        last_layer_activations = outputs.hidden_states[-1].cpu().numpy()
    
    # Project activations onto feature directions and take mean across sequence length
    feature_activations = np.dot(last_layer_activations.squeeze(), feature_directions.T)
    if len(feature_activations.shape) > 1:
        feature_activations = feature_activations.mean(axis=0)
    return feature_activations

# Test texts covering different concepts
test_texts = [
    "The cat sat on the mat",  # Simple sentence with objects
    "2 + 2 = 4",              # Mathematical expression
    "😊 Happy birthday! 🎂",   # Emojis and sentiment
    "def factorial(n):",       # Programming code
    "http://example.com",      # URL
    "She felt sad because"     # Emotion and causality
]

# Get activations for each text
all_activations = []
for text in test_texts:
    activations = get_feature_activations(text, model, tokenizer, feature_directions)
    all_activations.append(activations)

# Create activation heatmap
plt.figure(figsize=(15, 8))
sns.heatmap(
    np.stack(all_activations),
    xticklabels=range(feature_directions.shape[1]),
    yticklabels=test_texts,
    cmap='RdBu_r',
    center=0
)
plt.title('Feature Activations Across Different Inputs')
plt.xlabel('Feature Index')
plt.ylabel('Input Text')
plt.tight_layout()
plt.show()



## 3. Feature Interpretation

Let's analyze what different features might represent:

In [ ]:


def find_top_activating_tokens(feature_idx: int, n_tokens: int = 10) -> list[str]:
    """Find tokens that most strongly activate a given feature."""
    feature_weights = feature_directions[:, feature_idx]
    top_indices = np.argsort(feature_weights)[-n_tokens:]
    return [tokenizer.decode([i]) for i in top_indices]

def analyze_feature(feature_idx: int):
    """Analyze a specific feature's behavior."""
    print(f"\nAnalyzing Feature {feature_idx}")
    print("Top activating tokens:")
    top_tokens = find_top_activating_tokens(feature_idx)
    for i, token in enumerate(reversed(top_tokens), 1):
        print(f"{i}. {token}")
    
    # Get feature metadata
    if feature_idx in feature_metadata.index:
        meta = feature_metadata.loc[feature_idx]
        print(f"\nFeature type: {meta['type']}")
        print(f"Description: {meta['description']}")

# Analyze some interesting features
interesting_features = [0, 10, 20, 30, 40]  # Replace with actual interesting feature indices
for feature_idx in interesting_features:
    analyze_feature(feature_idx)



## 4. Feature Clustering

Let's group similar features together to understand broader patterns:

In [ ]:


from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Reduce dimensionality for visualization
pca = PCA(n_components=2)
feature_embeddings = pca.fit_transform(feature_directions.T)

# Cluster features
n_clusters = 5
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
clusters = kmeans.fit_predict(feature_directions.T)

# Visualize clusters
plt.figure(figsize=(12, 8))
scatter = plt.scatter(
    feature_embeddings[:, 0],
    feature_embeddings[:, 1],
    c=clusters,
    cmap='tab10',
    alpha=0.6
)
plt.title('Feature Clusters in 2D PCA Space')
plt.xlabel('First Principal Component')
plt.ylabel('Second Principal Component')
plt.colorbar(scatter, label='Cluster')
plt.show()

# Analyze cluster characteristics
for cluster_id in range(n_clusters):
    cluster_features = np.where(clusters == cluster_id)[0]
    print(f"\nCluster {cluster_id} Analysis:")
    print(f"Number of features: {len(cluster_features)}")
    
    # Get representative features
    center_idx = cluster_features[
        np.argmin(np.linalg.norm(
            feature_directions[:, cluster_features].T - 
            kmeans.cluster_centers_[cluster_id],
            axis=1
        ))
    ]
    print("Representative feature tokens:")
    analyze_feature(center_idx)



## 5. Finding Surprising Features

Let's look for features with unexpected or interesting behaviors:

In [ ]:


def find_surprising_features(n_features: int = 5) -> list[int]:
    """Find features with unusual activation patterns."""
    surprising_features = []
    
    # 1. Features with high sparsity
    activations_matrix = np.vstack(all_activations)
    sparsity = np.mean(activations_matrix == 0, axis=0)
    high_sparsity_features = np.argsort(sparsity)[-n_features:]
    
    # 2. Features with high variance
    variance = np.var(activations_matrix, axis=0)
    high_variance_features = np.argsort(variance)[-n_features:]
    
    # 3. Features with unusual activation distributions
    kurtosis = scipy.stats.kurtosis(activations_matrix, axis=0)
    unusual_dist_features = np.argsort(np.abs(kurtosis))[-n_features:]
    
    surprising_features = np.unique(np.concatenate([
        high_sparsity_features,
        high_variance_features,
        unusual_dist_features
    ]))
    
    return surprising_features.tolist()

# Find and analyze surprising features
surprising_features = find_surprising_features()
print("Surprising Features Analysis:")
for feature_idx in surprising_features:
    analyze_feature(feature_idx)
    
    # Plot activation distribution
    activations = np.vstack(all_activations)[:, feature_idx]
    plt.figure(figsize=(10, 4))
    plt.hist(activations[activations != 0], bins=50)
    plt.title(f'Feature {feature_idx} Activation Distribution')
    plt.xlabel('Activation Value')
    plt.ylabel('Count')
    plt.show()



## Discussion Points

1. Feature Interpretability
   - What types of features are most interpretable?
   - How do features combine to represent complex concepts?
   - What's the relationship between features and tokens?

2. Feature Organization
   - How are similar features grouped together?
   - What patterns emerge from feature clustering?
   - How does this relate to model behavior?

3. Surprising Discoveries
   - What unexpected features did you find?
   - How might these features contribute to model capabilities?
   - What does this tell us about model internals?

4. Implications for Interpretability
   - How useful are these features for understanding models?
   - What are the limitations of this analysis?
   - How could we improve feature extraction?